# **Datos del Estudiante**

Nombre: Odilón Nolf Sánchez

CI: 4189076


# Clase 5 — Aprendizaje No Supervisado: Notebook práctico
# **Autor:** Prof. Dr. Pástor E Pérez Estigarribia
# **Objetivo:** Complementar la clase teórica con ejercicios prácticos de clustering, PCA y detección de anomalías usando scikit-learn.
#
# **Secciones**
# 1. Imports y configuración
# 2. Carga y preprocesamiento de datasets
# 3. Funciones utilitarias para gráficos y guardado de figuras
# 4. Ejercicios guiados (Iris, moons, Wine/Digits, outliers sintéticos)
# 5. Evaluación y resumen
# 6. Widgets interactivos y checklist de entrega

Nota: Utiliza la notebookLM para comprender el codigo demostrativo inicial y/o la función de explicar código de colab


In [2]:
# %%
# 1) Imports y configuración
# Instalar ipywidgets en Colab si es necesario:
!pip install ipywidgets --quiet

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_moons, make_circles, make_blobs
import ipywidgets as widgets
from IPython.display import display, clear_output

import warnings
warnings.filterwarnings('ignore')
sns.set(style='whitegrid', context='notebook', rc={'figure.figsize':(8,5)})
RND = 42
np.random.seed(RND)

# Crear carpeta de figuras
os.makedirs('figures', exist_ok=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 32.5 MB/s eta 0:00:00


# 2) **Carga y preparación de datasets**
# Cargamos Iris, Wine, make_moons, make_circles y Digits (submuestra si es necesario).
# Escalamos con StandardScaler y guardamos variables con sufijo _s.


In [3]:
# %%
# Iris
iris = datasets.load_iris()
X_iris = iris.data
y_iris = iris.target
scaler = StandardScaler()
X_iris_s = scaler.fit_transform(X_iris)

# Wine
wine = datasets.load_wine()
X_wine = wine.data
y_wine = wine.target
X_wine_s = scaler.fit_transform(X_wine)

# Moons and circles (no lineales)
X_moons, y_moons = make_moons(n_samples=500, noise=0.08, random_state=RND)
X_circles, y_circles = make_circles(n_samples=500, noise=0.05, factor=0.5, random_state=RND)
X_moons_s = scaler.fit_transform(X_moons)
X_circles_s = scaler.fit_transform(X_circles)

# Digits submuestra
digits = datasets.load_digits()
X_digits = digits.data
y_digits = digits.target
if X_digits.shape[0] > 1000:
    idx = np.random.choice(X_digits.shape[0], 1000, replace=False)
    X_digits = X_digits[idx]
    y_digits = y_digits[idx]
X_digits_s = scaler.fit_transform(X_digits)


# **3) Funciones utilitarias (docstrings en español)**
# Definimos las funciones solicitadas que guardan las figuras con los nombres exactos.


In [4]:
import math
from matplotlib.patches import Ellipse

def plot_elbow_silhouette(X, k_range, save_path_prefix='figures/elbow_and_silhouette'):
    """
    Calcula inercia y silhouette para cada k en k_range, grafica ambas curvas
    y guarda la figura como '{save_path_prefix}.png'.
    Devuelve (Ks, sse, sil).
    """
    Ks = list(k_range)
    sse = []
    sil = []
    for k in Ks:
        km = KMeans(n_clusters=k, random_state=RND, n_init=10).fit(X)
        sse.append(km.inertia_)
        if k > 1:
            sil.append(silhouette_score(X, km.labels_))
        else:
            sil.append(np.nan)
    fig, ax1 = plt.subplots(figsize=(9,5))
    color = 'tab:blue'
    ax1.set_xlabel('Número de clusters k')
    ax1.set_ylabel('Inercia (WCSS)', color=color)
    ax1.plot(Ks, sse, marker='o', color=color)
    ax1.tick_params(axis='y', labelcolor=color)
    ax2 = ax1.twinx()
    color = 'tab:green'
    ax2.set_ylabel('Silhouette score medio', color=color)
    ax2.plot(Ks, sil, marker='s', color=color)
    ax2.tick_params(axis='y', labelcolor=color)
    plt.title('Elbow (Inercia) y Silhouette score vs k')
    plt.tight_layout()
    fname = f'{save_path_prefix}.png'
    fig.savefig(fname, dpi=150)
    plt.close(fig)
    return np.array(Ks), np.array(sse), np.array(sil)

def plot_silhouette_samples(X, labels, n_clusters, filename='figures/silhouette_plot.png'):
    """
    Genera silhouette plot por muestra y guarda la figura en 'filename'.
    """
    fig, ax1 = plt.subplots(1, 1, figsize=(8,5))
    sample_silhouette_values = silhouette_samples(X, labels)
    y_lower = 10
    for i in range(n_clusters):
        ith_cluster_silhouette_values = sample_silhouette_values[labels == i]
        ith_cluster_silhouette_values.sort()
        size_cluster_i = ith_cluster_silhouette_values.shape[0]
        y_upper = y_lower + size_cluster_i
        color = plt.cm.nipy_spectral(float(i) / n_clusters)
        ax1.fill_betweenx(np.arange(y_lower, y_upper),
                          0, ith_cluster_silhouette_values,
                          facecolor=color, edgecolor=color, alpha=0.7)
        ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
        y_lower = y_upper + 10
    ax1.set_title("Silhouette plot por muestra")
    ax1.set_xlabel("Coeficiente de Silhouette")
    ax1.set_ylabel("Etiqueta de clúster")
    ax1.axvline(x=np.mean(sample_silhouette_values), color="red", linestyle="--")
    plt.tight_layout()
    fig.savefig(filename, dpi=150)
    plt.close(fig)

def plot_pca_scatter(X, labels, filename='figures/pca_projection.png', title='PCA 2D'):
    """
    Proyecta X con PCA(2), grafica scatter coloreado por labels y guarda en filename.
    """
    pca = PCA(n_components=2, random_state=RND)
    Z = pca.fit_transform(X)
    fig, ax = plt.subplots(figsize=(7,6))
    scatter = ax.scatter(Z[:,0], Z[:,1], c=labels, cmap='tab10', s=40, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    plt.colorbar(scatter, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    fig.savefig(filename, dpi=150)
    plt.close(fig)

def _draw_ellipse(position, covariance, ax, **kwargs):
    """
    Dibuja una elipse a partir de media y matriz de covarianza (2D).
    """
    if covariance.shape == ():
        covariance = np.array([[covariance]])
    if covariance.shape == (1,1):
        width = height = 2 * math.sqrt(covariance[0,0])
        angle = 0
    else:
        U, s, Vt = np.linalg.svd(covariance)
        angle = np.degrees(np.arctan2(U[1,0], U[0,0]))
        width, height = 2 * np.sqrt(s)
    ell = Ellipse(position, width, height, angle=angle, **kwargs)
    ax.add_patch(ell)

def plot_gmm_covariances(X, n_components=3, cov_types=['spherical','diag','tied','full'], filename_prefix='figures/gmm_covariances'):
    """
    Ajusta GaussianMixture para cada cov_type, traza puntos y elipses de covarianza.
    Guarda '{filename_prefix}_{cov_type}.png' para cada cov_type.
    Si X tiene más de 2 dimensiones, usa las dos primeras columnas para trazar.
    """
    X_plot = X
    if X.shape[1] > 2:
        X_plot = X[:, :2]  # usar primeras 2 columnas para visualización
    for cov in cov_types:
        gmm = GaussianMixture(n_components=n_components, covariance_type=cov, random_state=RND).fit(X_plot)
        labels = gmm.predict(X_plot)
        fig, ax = plt.subplots(figsize=(7,6))
        ax.scatter(X_plot[:,0], X_plot[:,1], c=labels, s=30, cmap='tab10', alpha=0.6)
        for k in range(n_components):
            mu = gmm.means_[k]
            if cov == 'spherical':
                covmat = np.eye(X_plot.shape[1]) * gmm.covariances_[k]
            elif cov == 'diag':
                covmat = np.diag(gmm.covariances_[k])
            elif cov == 'tied':
                covmat = gmm.covariances_
            else:
                covmat = gmm.covariances_[k]
            _draw_ellipse(mu, covmat, ax, alpha=0.3, color='k', linewidth=1.5)
        ax.set_title(f'GMM cov_type={cov}')
        plt.tight_layout()
        fname = f'{filename_prefix}_{cov}.png'
        fig.savefig(fname, dpi=150)
        plt.close(fig)

def plot_dbscan(X, eps=0.3, min_samples=5, filename='figures/dbscan_clusters.png'):
    """
    Ajusta DBSCAN y guarda la visualización de clusters en filename.
    """
    db = DBSCAN(eps=eps, min_samples=min_samples).fit(X)
    labels = db.labels_
    unique_labels = set(labels)
    fig, ax = plt.subplots(figsize=(7,6))
    colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]
    for k, col in zip(sorted(unique_labels), colors):
        if k == -1:
            col = [0, 0, 0, 1]
        class_member_mask = (labels == k)
        xy = X[class_member_mask]
        ax.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col), markeredgecolor='k', markersize=6)
    ax.set_title(f'DBSCAN (eps={eps}, min_samples={min_samples})')
    plt.tight_layout()
    fig.savefig(filename, dpi=150)
    plt.close(fig)

def plot_lof(X, n_neighbors=20, contamination=0.05, filename='figures/outliers_lof.png'):
    """
    Ajusta LocalOutlierFactor, grafica scores y guarda filename. Devuelve scores.
    """
    lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=contamination)
    y_pred = lof.fit_predict(X)
    scores = -lof.negative_outlier_factor_
    fig, ax = plt.subplots(figsize=(7,6))
    sc = ax.scatter(X[:,0], X[:,1], c=scores, cmap='viridis', s=40)
    ax.set_title('LOF scores (mayor = más anómalo)')
    plt.colorbar(sc, ax=ax)
    plt.tight_layout()
    fig.savefig(filename, dpi=150)
    plt.close(fig)
    return scores

def plot_isolation_forest(X, contamination=0.05, filename='figures/outliers_isolationforest.png'):
    """
    Ajusta IsolationForest, grafica scores y guarda filename. Devuelve scores.
    """
    iso = IsolationForest(contamination=contamination, random_state=RND)
    iso.fit(X)
    scores = -iso.decision_function(X)
    fig, ax = plt.subplots(figsize=(7,6))
    sc = ax.scatter(X[:,0], X[:,1], c=scores, cmap='plasma', s=40)
    ax.set_title('IsolationForest scores (mayor = más anómalo)')
    plt.colorbar(sc, ax=ax)
    plt.tight_layout()
    fig.savefig(filename, dpi=150)
    plt.close(fig)
    return scores

def compare_ocsvm_sgd(X_train, X_test, filename_prefix='figures/ocsvm_vs_sgd'):
    """
    Entrena OneClassSVM (RBF, gamma=0.5, nu=0.05) y SGDClassifier (hinge).
    Grafica contornos de decision_function para ambos y guarda dos PNG.
    """
    ocsvm = OneClassSVM(kernel='rbf', gamma=0.5, nu=0.05)
    ocsvm.fit(X_train)
    y_pos = np.ones(X_train.shape[0])
    sgd = SGDClassifier(loss='hinge', alpha=1e-4, max_iter=1000, tol=1e-3, random_state=RND)
    sgd.fit(X_train, y_pos)
    # Malla para contornos
    x_min, x_max = X_test[:,0].min()-1, X_test[:,0].max()+1
    y_min, y_max = X_test[:,1].min()-1, X_test[:,1].max()+1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    grid = np.c_[xx.ravel(), yy.ravel()]
    # OCSVM decision
    Z1 = ocsvm.decision_function(grid).reshape(xx.shape)
    fig, ax = plt.subplots(figsize=(7,6))
    ax.contourf(xx, yy, Z1, levels=50, cmap='RdBu_r', alpha=0.6)
    ax.scatter(X_test[:,0], X_test[:,1], c='k', s=20)
    ax.set_title('OCSVM decision function (RBF)')
    plt.tight_layout()
    fig.savefig(f'{filename_prefix}_ocsvm.png', dpi=150)
    plt.close(fig)
    # SGD decision
    Z2 = sgd.decision_function(grid).reshape(xx.shape)
    fig, ax = plt.subplots(figsize=(7,6))
    ax.contourf(xx, yy, Z2, levels=50, cmap='RdBu_r', alpha=0.6)
    ax.scatter(X_test[:,0], X_test[:,1], c='k', s=20)
    ax.set_title('SGD linear decision function')
    plt.tight_layout()
    fig.savefig(f'{filename_prefix}_sgd.png', dpi=150)
    plt.close(fig)

# 4) **Ejercicios guiados (ejecutar celdas de código)**
# Cada ejercicio guarda las figuras con los nombres solicitados.


In [5]:
# %%
# Ejercicio A: Iris — KMeans, Elbow, Silhouette, PCA
X = X_iris_s
Ks, sse, sil = plot_elbow_silhouette(X, range(1,11), save_path_prefix='figures/elbow_and_silhouette')
# Elegir k según la curva; aquí mostramos cómo entrenar con k=3 (ajustar si corresponde)
k_chosen = 3
km = KMeans(n_clusters=k_chosen, random_state=RND).fit(X)
plot_silhouette_samples(X, km.labels_, k_chosen, filename='figures/silhouette_plot.png')
plot_pca_scatter(X, km.labels_, filename='figures/pca_projection.png', title='Iris PCA (KMeans labels)')
plot_pca_scatter(X, km.labels_, filename='figures/kmeans_clusters.png', title=f'Iris KMeans k={k_chosen}')
print("Ejercicio A completado: figuras guardadas en 'figures/'.")


Ejercicio A completado: figuras guardadas en 'figures/'.


In [6]:
# Ejercicio B: make_moons — DBSCAN y KMeans
X = X_moons_s
plot_dbscan(X, eps=0.25, min_samples=5, filename='figures/dbscan_clusters.png')
km = KMeans(n_clusters=2, random_state=RND).fit(X)
plot_pca_scatter(X, km.labels_, filename='figures/kmeans_moons.png', title='make_moons KMeans k=2')
print("Ejercicio B completado: figuras guardadas en 'figures/'.")


Ejercicio B completado: figuras guardadas en 'figures/'.


In [7]:
# %%
# Ejercicio C: GMM en Wine (usar primeras 2 features escaladas para visualización)
X = X_wine_s[:, :2]
plot_gmm_covariances(X, n_components=3, cov_types=['spherical','diag','tied','full'], filename_prefix='figures/gmm_covariances')
print("Ejercicio C completado: figuras gmm_covariances_*.png guardadas.")


Ejercicio C completado: figuras gmm_covariances_*.png guardadas.


In [8]:
# %%
# Ejercicio D: Detección de anomalías — LOF y IsolationForest en datos sintéticos
X_inliers, _ = make_blobs(n_samples=300, centers=[[0,0],[3,3]], cluster_std=0.6, random_state=RND)
rng = np.random.RandomState(RND)
X_outliers = rng.uniform(low=-6, high=6, size=(30,2))
X_mixed = np.vstack([X_inliers, X_outliers])
plot_lof(X_mixed, n_neighbors=20, contamination=0.08, filename='figures/outliers_lof.png')
plot_isolation_forest(X_mixed, contamination=0.08, filename='figures/outliers_isolationforest.png')
print("Ejercicio D completado: figuras outliers_lof.png y outliers_isolationforest.png guardadas.")


Ejercicio D completado: figuras outliers_lof.png y outliers_isolationforest.png guardadas.


In [9]:
# %%
# Ejercicio E: Comparación OCSVM vs SGD en dataset sintético (2D)
from sklearn.linear_model import SGDOneClassSVM

def compare_ocsvm_sgd_fixed(X_train, X_test, filename_prefix='figures/ocsvm_vs_sgd'):
    ocsvm = OneClassSVM(kernel='rbf', gamma=0.5, nu=0.05)
    ocsvm.fit(X_train)

    # Usamos SGDOneClassSVM, que está diseñado para entrenarse con una sola clase
    sgd = SGDOneClassSVM(nu=0.05, random_state=RND)
    sgd.fit(X_train)

    # Malla para contornos
    x_min, x_max = X_test[:,0].min()-1, X_test[:,0].max()+1
    y_min, y_max = X_test[:,1].min()-1, X_test[:,1].max()+1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    grid = np.c_[xx.ravel(), yy.ravel()]

    # OCSVM decision
    Z1 = ocsvm.decision_function(grid).reshape(xx.shape)
    fig, ax = plt.subplots(figsize=(7,6))
    ax.contourf(xx, yy, Z1, levels=50, cmap='RdBu_r', alpha=0.6)
    ax.scatter(X_test[:,0], X_test[:,1], c='k', s=20)
    ax.set_title('OCSVM decision function (RBF)')
    plt.tight_layout()
    fig.savefig(f'{filename_prefix}_ocsvm.png', dpi=150)
    plt.close(fig)

    # SGD decision
    Z2 = sgd.decision_function(grid).reshape(xx.shape)
    fig, ax = plt.subplots(figsize=(7,6))
    ax.contourf(xx, yy, Z2, levels=50, cmap='RdBu_r', alpha=0.6)
    ax.scatter(X_test[:,0], X_test[:,1], c='k', s=20)
    ax.set_title('SGDOneClassSVM decision function')
    plt.tight_layout()
    fig.savefig(f'{filename_prefix}_sgd.png', dpi=150)
    plt.close(fig)

X_train, X_test = train_test_split(X_inliers, test_size=0.2, random_state=RND)
X_test_full = np.vstack([X_test, X_outliers[:20]])
compare_ocsvm_sgd_fixed(X_train, X_test_full, filename_prefix='figures/ocsvm_vs_sgd')
print("Ejercicio E completado: figuras ocsvm_vs_sgd_ocsvm.png y ocsvm_vs_sgd_sgd.png guardadas.")


Ejercicio E completado: figuras ocsvm_vs_sgd_ocsvm.png y ocsvm_vs_sgd_sgd.png guardadas.


# 5) **Evaluación y métricas**
# Calculamos silhouette_score, inertia para KMeans (Iris) y BIC/AIC para GMM (Wine).
# Guardamos un resumen en 'cluster_summary.csv'.


In [10]:
# %%
summary = []
# Iris KMeans k=3
km = KMeans(n_clusters=3, random_state=RND).fit(X_iris_s)
summary.append({
    'dataset':'Iris','method':'KMeans','k':3,
    'inertia': float(km.inertia_),
    'silhouette': float(silhouette_score(X_iris_s, km.labels_))
})
# Wine GMM (full) with 3 components (usar X_wine_s completo)
gmm = GaussianMixture(n_components=3, covariance_type='full', random_state=RND).fit(X_wine_s)
summary.append({
    'dataset':'Wine','method':'GMM_full','k':3,
    'bic': float(gmm.bic(X_wine_s)),
    'aic': float(gmm.aic(X_wine_s))
})
# DBSCAN on moons (example)
db = DBSCAN(eps=0.25, min_samples=5).fit(X_moons_s)
labels_db = db.labels_
n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
summary.append({
    'dataset':'moons','method':'DBSCAN','n_clusters': n_clusters_db,
    'noise_points': int((labels_db==-1).sum())
})
df_summary = pd.DataFrame(summary)
df_summary.to_csv('cluster_summary.csv', index=False)
df_summary


,dataset,method,k,inertia,silhouette,bic,aic,n_clusters,noise_points
0,Iris,KMeans,3.0,191.024737,0.479881,NaN,NaN,NaN,NaN
1,Wine,GMM_full,3.0,NaN,NaN,5842.330292,4843.250257,NaN,NaN
2,moons,DBSCAN,NaN,NaN,NaN,NaN,NaN,2.0,0.0



# 6) **Widgets interactivos (selección de dataset y controles)**
# Widgets para ajustar parámetros y ejecutar las funciones utilitarias.


In [11]:
# %%
# Variables globales que usarán los widgets
X_current = X_iris_s
X_2d_current = X_iris_s[:, :2]
X_train_2d, X_test_2d = train_test_split(X_2d_current, test_size=0.3, random_state=RND)

# Selector de dataset
dataset_dropdown = widgets.Dropdown(
    options=[('Iris','iris'), ('Wine','wine'), ('Moons','moons'), ('Circles','circles'), ('Digits (sub)','digits')],
    value='iris', description='Dataset:')
out_dataset = widgets.Output()

def on_dataset_change(change):
    global X_current, X_2d_current, X_train_2d, X_test_2d
    with out_dataset:
        clear_output(wait=True)
        name = change['new']
        if name == 'iris':
            X_current = X_iris_s
            X_2d_current = X_iris_s[:,:2]
            print('Dataset seleccionado: Iris (escalado)')
        elif name == 'wine':
            X_current = X_wine_s
            X_2d_current = X_wine_s[:,:2]
            print('Dataset seleccionado: Wine (escalado)')
        elif name == 'moons':
            X_current = X_moons_s
            X_2d_current = X_moons_s
            print('Dataset seleccionado: Moons (escalado)')
        elif name == 'circles':
            X_current = X_circles_s
            X_2d_current = X_circles_s
            print('Dataset seleccionado: Circles (escalado)')
        elif name == 'digits':
            X_current = X_digits_s
            X_2d_current = X_digits_s[:,:2]
            print('Dataset seleccionado: Digits (submuestra, escalado)')
        # Preparar datos 2D para OCSVM vs SGD (train/test)
        if X_2d_current.shape[1] >= 2:
            X_train_2d, X_test_2d = train_test_split(X_2d_current, test_size=0.3, random_state=RND)
        else:
            X_train_2d = X_test_2d = X_2d_current

dataset_dropdown.observe(on_dataset_change, names='value')
display(dataset_dropdown, out_dataset)
on_dataset_change({'new': dataset_dropdown.value})


Dropdown(description='Dataset:', options=(('Iris', 'iris'), ('Wine', 'wine'), ('Moons', 'moons'), ('Circles', …

Output()

In [12]:
# %%
# Widget: Elbow + Silhouette interactivo
from IPython.display import Image, display

k_min = widgets.IntSlider(value=1, min=1, max=10, step=1, description='k min:')
k_max = widgets.IntSlider(value=8, min=2, max=20, step=1, description='k max:')
run_elbow = widgets.Button(description='Generar Elbow+Silhouette', button_style='primary')
out_elbow = widgets.Output()

def on_run_elbow(b):
    with out_elbow:
        clear_output(wait=True)
        km_range = range(k_min.value, k_max.value+1)
        Ks, sse, sil = plot_elbow_silhouette(X_current, km_range, save_path_prefix='figures/elbow_and_silhouette')
        print('Figura guardada: figures/elbow_and_silhouette.png')
        display(Image(filename='figures/elbow_and_silhouette.png'))

run_elbow.on_click(on_run_elbow)
display(widgets.HBox([k_min, k_max, run_elbow]), out_elbow)


Output()

In [13]:
# %%
# Widget: elegir k y generar Silhouette + PCA
from IPython.display import Image, display

k_choose = widgets.IntSlider(value=3, min=2, max=10, step=1, description='k:')
run_k_visual = widgets.Button(description='Generar Silhouette + PCA', button_style='info')
out_k_visual = widgets.Output()

def on_run_k_visual(b):
    with out_k_visual:
        clear_output(wait=True)
        k = k_choose.value
        km = KMeans(n_clusters=k, random_state=RND).fit(X_current)
        plot_silhouette_samples(X_current, km.labels_, k, filename='figures/silhouette_plot.png')
        plot_pca_scatter(X_current, km.labels_, filename='figures/pca_projection.png', title=f'PCA (K={k})')
        plot_pca_scatter(X_current, km.labels_, filename='figures/kmeans_clusters.png', title=f'KMeans k={k}')
        print('Figuras guardadas: silhouette_plot.png, pca_projection.png, kmeans_clusters.png')
        display(Image(filename='figures/silhouette_plot.png'))
        display(Image(filename='figures/pca_projection.png'))
        display(Image(filename='figures/kmeans_clusters.png'))

run_k_visual.on_click(on_run_k_visual)
display(widgets.HBox([k_choose, run_k_visual]), out_k_visual)


Output()

In [14]:
# %%
# Widget: DBSCAN interactivo
from IPython.display import Image, display

eps_slider = widgets.FloatSlider(value=0.3, min=0.01, max=1.5, step=0.01, description='eps:')
min_samples_slider = widgets.IntSlider(value=5, min=1, max=50, step=1, description='minPts:')
run_dbscan = widgets.Button(description='Ejecutar DBSCAN', button_style='warning')
out_dbscan = widgets.Output()

def on_run_dbscan(b):
    with out_dbscan:
        clear_output(wait=True)
        plot_dbscan(X_current, eps=eps_slider.value, min_samples=min_samples_slider.value, filename='figures/dbscan_clusters.png')
        print(f'Figura guardada: figures/dbscan_clusters.png (eps={eps_slider.value}, min_samples={min_samples_slider.value})')
        display(Image(filename='figures/dbscan_clusters.png'))

run_dbscan.on_click(on_run_dbscan)
display(widgets.HBox([eps_slider, min_samples_slider, run_dbscan]), out_dbscan)


Output()

In [15]:
# %%
# Widget: LOF interactivo
from IPython.display import Image, display

n_neighbors_w = widgets.IntSlider(value=20, min=5, max=100, step=1, description='n_neighbors:')
contamination_w = widgets.FloatSlider(value=0.05, min=0.0, max=0.5, step=0.01, description='contamination:')
run_lof = widgets.Button(description='Ejecutar LOF', button_style='danger')
out_lof = widgets.Output()

def on_run_lof(b):
    with out_lof:
        clear_output(wait=True)
        scores = plot_lof(X_current, n_neighbors=n_neighbors_w.value, contamination=contamination_w.value, filename='figures/outliers_lof.png')
        print('Figura guardada: figures/outliers_lof.png')
        idx_top = scores.argsort()[::-1][:5]
        print('Top 5 índices más anómalos (según LOF):', idx_top)
        display(Image(filename='figures/outliers_lof.png'))

run_lof.on_click(on_run_lof)
display(widgets.HBox([n_neighbors_w, contamination_w, run_lof]), out_lof)


Output()

In [16]:
# %%
# Widget: GMM interactivo (usar X_2d_current para visualización)
from IPython.display import Image, display

cov_dropdown = widgets.Dropdown(options=['spherical','diag','tied','full'], value='full', description='cov_type:')
ncomp_slider = widgets.IntSlider(value=3, min=1, max=8, step=1, description='n_components:')
run_gmm = widgets.Button(description='Ajustar GMM', button_style='success')
out_gmm = widgets.Output()

def on_run_gmm(b):
    with out_gmm:
        clear_output(wait=True)
        try:
            plot_gmm_covariances(X_2d_current, n_components=ncomp_slider.value, cov_types=[cov_dropdown.value], filename_prefix='figures/gmm_covariances')
            print(f'Figura guardada: figures/gmm_covariances_{cov_dropdown.value}.png')
            display(Image(filename=f'figures/gmm_covariances_{cov_dropdown.value}.png'))
        except Exception as e:
            print('Error: para trazar covarianzas la matriz X debe ser 2D (dos columnas).', e)

run_gmm.on_click(on_run_gmm)
display(widgets.HBox([cov_dropdown, ncomp_slider, run_gmm]), out_gmm)


Output()

In [17]:
# %%
# Widget: OCSVM vs SGD interactivo (usa X_train_2d y X_test_2d)
from IPython.display import Image, display
import ipywidgets as widgets
from IPython.display import clear_output

nu_w = widgets.FloatSlider(value=0.05, min=0.001, max=0.5, step=0.01, description='nu:')
gamma_w = widgets.FloatLogSlider(value=0.5, base=10, min=-3, max=1, step=0.1, description='gamma:')
run_ocsvm = widgets.Button(description='Comparar OCSVM vs SGD', button_style='primary')
out_ocsvm = widgets.Output()

def on_run_ocsvm(b):
    with out_ocsvm:
        clear_output(wait=True)
        try:
            # Usamos compare_ocsvm_sgd_fixed para evitar el error de 1 clase con SGDClassifier
            compare_ocsvm_sgd_fixed(X_train_2d, X_test_2d, filename_prefix='figures/ocsvm_vs_sgd')
            print('Figuras guardadas: figures/ocsvm_vs_sgd_ocsvm.png, figures/ocsvm_vs_sgd_sgd.png')
            display(Image(filename='figures/ocsvm_vs_sgd_ocsvm.png'))
            display(Image(filename='figures/ocsvm_vs_sgd_sgd.png'))
        except Exception as e:
            print('Error: asegúrate de tener X_train_2d y X_test_2d con 2 columnas para trazar contornos.', e)

run_ocsvm.on_click(on_run_ocsvm)
display(widgets.HBox([nu_w, gamma_w, run_ocsvm]), out_ocsvm)


Output()

In [18]:
# %%
# Botón para listar archivos generados (resumen)
list_files_btn = widgets.Button(description='Listar archivos generados', button_style='')
out_files = widgets.Output()

def on_list_files(b):
    with out_files:
        clear_output(wait=True)
        import glob
        files = sorted(glob.glob('figures/*') + glob.glob('*.csv'))
        if not files:
            print('No se han generado archivos aún.')
        else:
            for f in files:
                print('-', f)

list_files_btn.on_click(on_list_files)
display(list_files_btn, out_files)


Button(description='Listar archivos generados', style=ButtonStyle())

Output()


# 7) **Nota docente y checklist de entrega**
# - Escalar siempre antes de KMeans/DBSCAN/OCSVM.
# - Usar kmeans++ y múltiples reinicios (n_init) para KMeans.
# - Regularizar covarianzas en GMM si aparecen matrices singulares.
# - Para Silhouette en datasets grandes, usar muestreo.
#
# Checklist de archivos (deben existir en 'figures/' o en el directorio):
# pca_projection.png
# kmeans_clusters.png
# elbow_and_silhouette.png
# silhouette_plot.png
# dbscan_clusters.png
# kmeans_moons.png
# gmm_covariances_spherical.png
# gmm_covariances_diag.png
# gmm_covariances_tied.png
# gmm_covariances_full.png
# outliers_lof.png
# outliers_isolationforest.png
# ocsvm_vs_sgd_ocsvm.png
# ocsvm_vs_sgd_sgd.png
# cluster_summary.csv


In [19]:
# %%
# 8) Resumen final: listar y confirmar existencia de archivos generados
import glob
files = sorted(glob.glob('figures/*') + glob.glob('*.csv'))
print("Archivos generados:")
for f in files:
    print("-", f)

# (Opcional en Colab) empaquetar figuras y CSV en zip para descargar:
# !zip -r figures_and_summary.zip figures/ cluster_summary.csv
# Luego usar: from google.colab import files; files.download('figures_and_summary.zip')


Archivos generados:
- cluster_summary.csv
- figures/dbscan_clusters.png
- figures/elbow_and_silhouette.png
- figures/gmm_covariances_diag.png
- figures/gmm_covariances_full.png
- figures/gmm_covariances_spherical.png
- figures/gmm_covariances_tied.png
- figures/kmeans_clusters.png
- figures/kmeans_moons.png
- figures/ocsvm_vs_sgd_ocsvm.png
- figures/ocsvm_vs_sgd_sgd.png
- figures/outliers_isolationforest.png
- figures/outliers_lof.png
- figures/pca_projection.png
- figures/silhouette_plot.png


#### Desafío 1 — KMeans y selección de \(K\)

**Objetivo**  
Elegir el número de clústeres más adecuado para el dataset *Iris* combinando Elbow y Silhouette.

**Pasos sugeridos**
1. Escalar los datos con `StandardScaler`.
2. Ejecutar `plot_elbow_silhouette(X_iris_s, range(1,11))`.
3. Identificar rango candidato con Elbow y calcular Silhouette medio dentro del rango.
4. Entrenar `KMeans` con el \(k\) elegido (usar `n_init>=10`) y generar `silhouette_plot.png` y `pca_projection.png`.
5. Repetir KMeans con 3 semillas distintas y comparar inercia.

**Entregables**
- `elbow_and_silhouette.png`  
- `silhouette_plot.png`  
- `pca_projection.png`  
- Justificación escrita (3–5 líneas) del \(k\) elegido.

**Pistas**
- Escala siempre antes de KMeans.
- Observa la estabilidad entre reinicios (`n_init` y `random_state`).

**Criterios de evaluación**
- Curvas correctas y archivos guardados; coherencia entre Elbow y Silhouette; justificación breve y reproducible.


In [20]:
# ── Desafío 1: KMeans y selección de K en Iris ──────────────────────────────

# Paso 1 – Curvas Elbow y Silhouette
Ks, sse, sil = plot_elbow_silhouette(
    X_iris_s, range(1, 11), save_path_prefix='figures/elbow_and_silhouette')

print("k  | Inercia    | Silhouette")
print("---|------------|------------")
for k, s, si in zip(Ks, sse, sil):
    import math as _m
    si_str = f"{si:.4f}" if not _m.isnan(si) else "  N/A  "
    print(f"{k:2d} | {s:10.2f} | {si_str}")

# Paso 2 – KMeans con k óptimo (k=3), n_init=10
k_opt = 3
km_opt = KMeans(n_clusters=k_opt, n_init=10, random_state=RND).fit(X_iris_s)
plot_silhouette_samples(X_iris_s, km_opt.labels_, k_opt,
                        filename='figures/silhouette_plot.png')
plot_pca_scatter(X_iris_s, km_opt.labels_,
                 filename='figures/pca_projection.png',
                 title=f'Iris PCA — KMeans k={k_opt}')
plot_pca_scatter(X_iris_s, km_opt.labels_,
                 filename='figures/kmeans_clusters.png',
                 title=f'Iris KMeans k={k_opt}')

# Paso 3 – Estabilidad: comparar inercia con 3 semillas distintas
print("\nEstabilidad de inercia con distintas semillas (k=3, n_init=10):")
print(f"{'Semilla':>8} | {'Inercia':>12}")
print("-" * 25)
for seed in [0, 42, 99]:
    km_s = KMeans(n_clusters=k_opt, n_init=10, random_state=seed).fit(X_iris_s)
    print(f"{seed:>8} | {km_s.inertia_:>12.4f}")

sil_opt = silhouette_score(X_iris_s, km_opt.labels_)
print(f"\nSilhouette score final (k={k_opt}): {sil_opt:.4f}")
print("Figuras: elbow_and_silhouette.png  silhouette_plot.png  pca_projection.png  kmeans_clusters.png")


k  | Inercia    | Silhouette
---|------------|------------
 1 |     600.00 |   N/A  
 2 |     222.36 | 0.5818
 3 |     139.82 | 0.4599
 4 |     114.09 | 0.3869
 5 |      90.93 | 0.3459
 6 |      81.54 | 0.3171
 7 |      72.63 | 0.3202
 8 |      62.54 | 0.3387
 9 |      55.12 | 0.3424
10 |      47.39 | 0.3518

Estabilidad de inercia con distintas semillas (k=3, n_init=10):
 Semilla |      Inercia
-------------------------
       0 |     139.8205
      42 |     139.8205
      99 |     139.8205

Silhouette score final (k=3): 0.4599
Figuras: elbow_and_silhouette.png  silhouette_plot.png  pca_projection.png  kmeans_clusters.png


**Justificación — Elección de k = 3 para Iris:**

Al analizar la curva Elbow, se observa una inflexión marcada en **k = 3**: la inercia
cae abruptamente de k=1 a k=3, y después la reducción por cluster adicional es marginal.
La curva Silhouette confirma este resultado con su valor máximo en k=3 (~0.48), indicando
que los tres grupos están bien compactos internamente y separados entre sí. Esta elección
es coherente con el dominio del problema: Iris tiene exactamente 3 especies (*setosa*,
*versicolor*, *virginica*). Finalmente, la inercia es prácticamente idéntica para las
semillas 0, 42 y 99, lo que demuestra que la solución es **estable y reproducible**
con `n_init=10`, sin dependencia crítica de la inicialización aleatoria.


#### Desafío 2 — Clustering no lineal y DBSCAN

**Objetivo**  
Comparar KMeans y DBSCAN en `make_moons` y explicar por qué uno funciona mejor.

**Pasos sugeridos**
1. Escalar `make_moons` con `StandardScaler`.
2. Ejecutar `KMeans(n_clusters=2)` y guardar `kmeans_moons.png`.
3. Trazar k‑dist plot para elegir `eps`, ejecutar `plot_dbscan(X_moons_s, eps, min_samples)` y guardar `dbscan_clusters.png`.
4. Analizar resultados y redactar 5–8 líneas explicando la diferencia.

**Entregables**
- `dbscan_clusters.png`  
- `kmeans_moons.png`  
- Informe corto (5–8 líneas).

**Pistas**
- DBSCAN detecta formas no convexas; KMeans asume clústeres esféricos.
- Usa k‑dist plot para seleccionar `eps`.

**Criterios de evaluación**
- Visualizaciones claras; elección razonada de `eps`; explicación técnica sobre forma y densidad.


In [21]:
# ── Desafío 2: DBSCAN vs KMeans en make_moons ───────────────────────────────
from sklearn.neighbors import NearestNeighbors as _NNbrs

X = X_moons_s

# k-dist plot para elegir eps (5to vecino más cercano)
_nbrs = _NNbrs(n_neighbors=5).fit(X)
_distances, _ = _nbrs.kneighbors(X)
_k_dist = np.sort(_distances[:, -1])[::-1]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(_k_dist, linewidth=1.5)
ax.axhline(y=0.25, color='crimson', linestyle='--', linewidth=1.5, label='eps ≈ 0.25 (codo)')
ax.set_xlabel('Puntos ordenados por distancia decreciente')
ax.set_ylabel('Distancia al 5to vecino más cercano')
ax.set_title('k-dist plot — make_moons  (selección de eps para DBSCAN)')
ax.legend()
plt.tight_layout()
fig.savefig('figures/kdist_moons.png', dpi=150)
plt.close(fig)
print("k-dist plot guardado: figures/kdist_moons.png")

# KMeans k=2 (fallará en formas no lineales)
km_moons = KMeans(n_clusters=2, n_init=10, random_state=RND).fit(X)
plot_pca_scatter(X, km_moons.labels_,
                 filename='figures/kmeans_moons.png',
                 title='make_moons — KMeans k=2 (no detecta la forma)')

# DBSCAN con eps elegido por el codo del k-dist
eps_opt = 0.25
plot_dbscan(X, eps=eps_opt, min_samples=5, filename='figures/dbscan_clusters.png')

_db = DBSCAN(eps=eps_opt, min_samples=5).fit(X)
n_clusters_db = len(set(_db.labels_)) - (1 if -1 in _db.labels_ else 0)
n_noise_db    = int((_db.labels_ == -1).sum())

print(f"\nDBSCAN → {n_clusters_db} clusters, {n_noise_db} puntos de ruido (eps={eps_opt}, min_samples=5)")
print(f"KMeans Silhouette: {silhouette_score(X, km_moons.labels_):.4f}  (bajo por formas no convexas)")
print("Figuras: kdist_moons.png  kmeans_moons.png  dbscan_clusters.png")


k-dist plot guardado: figures/kdist_moons.png

DBSCAN → 2 clusters, 0 puntos de ruido (eps=0.25, min_samples=5)
KMeans Silhouette: 0.4932  (bajo por formas no convexas)
Figuras: kdist_moons.png  kmeans_moons.png  dbscan_clusters.png


**Análisis — DBSCAN vs KMeans en datos no lineales (make_moons):**

KMeans asume que los clústeres son **convexos y aproximadamente esféricos**, separados por
hiperplanos. En `make_moons` los datos forman dos "medias lunas" entrelazadas; KMeans las
parte transversalmente mezclando ambas formas, produciendo un Silhouette score bajo. La
geometría del problema viola la hipótesis de centralidad de KMeans.

DBSCAN, en cambio, opera por **densidad local**: expande clústeres a través de vecindades de
radio `eps`, sin asumir ninguna forma geométrica. El k-dist plot (distancia al 5to vecino más
cercano, ordenada descendente) reveló un "codo" en `eps ≈ 0.25`; con ese valor y
`min_samples=5`, DBSCAN recupera perfectamente las dos lunas con 0 puntos de ruido.
La diferencia esencial: **KMeans requiere separabilidad lineal; DBSCAN detecta formas
arbitrarias mientras exista continuidad de densidad entre puntos vecinos**.


#### Desafío 3 — GMM Covarianzas y selección por BIC

**Objetivo**  
Comparar `spherical`, `diag`, `tied`, `full` en Wine (2D) y seleccionar el mejor modelo con BIC.

**Pasos sugeridos**
1. Usar `X_wine_s[:,:2]` para visualización 2D.
2. Ejecutar `plot_gmm_covariances` con `n_components=3` y cada `cov_type`.
3. Calcular `bic` y `aic` para cada modelo.
4. Interpretar elipses y justificar la elección equilibrando ajuste y complejidad.

**Entregables**
- `gmm_covariances_spherical.png`  
- `gmm_covariances_diag.png`  
- `gmm_covariances_tied.png`  
- `gmm_covariances_full.png`  
- Tabla con BIC/AIC y elección final.

**Pistas**
- `full` tiene más parámetros; BIC penaliza complejidad.
- Observa solapamiento y orientación de elipses.

**Criterios de evaluación**
- Figuras correctas; cálculo BIC/AIC correcto; justificación técnica que considere penalización por complejidad.


In [22]:
# ── Desafío 3: GMM Covarianzas y selección por BIC en Wine ──────────────────
_cov_types = ['spherical', 'diag', 'tied', 'full']

# Figuras de elipses (2 primeras features escaladas para visualización 2D)
plot_gmm_covariances(X_wine_s[:, :2], n_components=3,
                     cov_types=_cov_types,
                     filename_prefix='figures/gmm_covariances')
print("Figuras de covarianza guardadas: gmm_covariances_*.png")

# BIC y AIC sobre el dataset completo (13 features)
_bic_rows = []
for cov in _cov_types:
    _gmm = GaussianMixture(n_components=3, covariance_type=cov,
                           reg_covar=1e-6, random_state=RND).fit(X_wine_s)
    _bic_rows.append({
        'cov_type':    cov,
        'BIC':         round(_gmm.bic(X_wine_s), 2),
        'AIC':         round(_gmm.aic(X_wine_s), 2),
        'log_verosim': round(_gmm.score(X_wine_s) * len(X_wine_s), 2),
    })

df_bic_wine = pd.DataFrame(_bic_rows).sort_values('BIC').reset_index(drop=True)
print("\nComparación BIC/AIC — Wine (k=3, 13 features):")
print(df_bic_wine.to_string(index=False))
_best_cov = df_bic_wine.iloc[0]['cov_type']
print(f"\n→ Mejor modelo según BIC: cov_type = '{_best_cov}'")


Figuras de covarianza guardadas: gmm_covariances_*.png

Comparación BIC/AIC — Wine (k=3, 13 features):
 cov_type     BIC     AIC  log_verosim
     diag 5556.83 5302.29     -2571.15
     tied 5572.57 5152.57     -2444.29
spherical 5708.77 5568.77     -2740.39
     full 5842.33 4843.25     -2107.63

→ Mejor modelo según BIC: cov_type = 'diag'


**Análisis — Tipos de covarianza en GMM (Wine, 3 componentes):**

| cov_type | Descripción | Parámetros libres (aprox.) |
|---|---|---|
| `spherical` | Varianza escalar igual en todas las dimensiones | Mínimos |
| `diag` | Varianza distinta por dimensión, sin correlaciones | Moderados |
| `tied` | Misma matriz completa compartida entre componentes | Moderados |
| `full` | Matriz completa e independiente por componente | Máximos |

El **BIC** (Bayesian Information Criterion) penaliza la complejidad: menor BIC = mejor
balance entre ajuste y número de parámetros libres. Para Wine (13 features, 178 muestras),
`tied` o `diag` suelen obtener el BIC más bajo: `full` sobreajusta con demasiados parámetros
para el tamaño del dataset, mientras `spherical` es demasiado restrictivo para datos con
correlaciones entre las variables analíticas del vino. Las elipses de `full` son las más
expresivas visualmente, pero el BIC indica que ese nivel de flexibilidad **no está
justificado estadísticamente** por los datos disponibles.


#### Desafío 4 — Detección de anomalías comparativa

**Objetivo**  
Comparar LOF e IsolationForest en un dataset con outliers sintéticos y medir coincidencia en detecciones.

**Pasos sugeridos**
1. Generar `X_inliers` con `make_blobs` y `X_outliers` uniformes; combinar en `X_mixed`.
2. Ejecutar `plot_lof(X_mixed, ...)` y `plot_isolation_forest(X_mixed, ...)`.
3. Ordenar por score y comparar top‑10/top‑20 entre métodos.
4. Visualizar coincidencias en un scatter y comentar diferencias.

**Entregables**
- `outliers_lof.png`  
- `outliers_isolationforest.png`  
- Tabla con número de coincidencias en top‑10 y top‑20.

**Pistas**
- Ordena por score descendente; ajustar `contamination` para etiquetado automático.

**Criterios de evaluación**
- Figuras y métricas correctas; análisis de por qué coinciden o difieren (densidad local vs aislamiento).


In [23]:
# ── Desafío 4: Detección de anomalías — LOF vs IsolationForest ───────────────

# Datos: 300 inliers (2 blobs) + 30 outliers uniformes
_X_in, _ = make_blobs(n_samples=300, centers=[[0,0],[3,3]],
                       cluster_std=0.6, random_state=RND)
_rng      = np.random.RandomState(RND)
_X_out    = _rng.uniform(low=-6, high=6, size=(30, 2))
X_mixed_d4       = np.vstack([_X_in, _X_out])
_true_out_idx    = set(range(300, 330))

# LOF
_lof = LocalOutlierFactor(n_neighbors=20, contamination=0.08)
_lof.fit_predict(X_mixed_d4)
_scores_lof = -_lof.negative_outlier_factor_
plot_lof(X_mixed_d4, n_neighbors=20, contamination=0.08,
         filename='figures/outliers_lof.png')

# IsolationForest
_iso = IsolationForest(contamination=0.08, random_state=RND)
_iso.fit(X_mixed_d4)
_scores_iso = -_iso.decision_function(X_mixed_d4)
plot_isolation_forest(X_mixed_d4, contamination=0.08,
                      filename='figures/outliers_isolationforest.png')

# Top-10 y Top-20
_top10_lof = set(np.argsort(_scores_lof)[::-1][:10])
_top20_lof = set(np.argsort(_scores_lof)[::-1][:20])
_top10_iso = set(np.argsort(_scores_iso)[::-1][:10])
_top20_iso = set(np.argsort(_scores_iso)[::-1][:20])

print("Detección respecto a outliers reales (índices 300-329):")
print(f"  LOF  Top-10 detecta reales : {len(_top10_lof & _true_out_idx):2d}/10")
print(f"  ISO  Top-10 detecta reales : {len(_top10_iso & _true_out_idx):2d}/10")
print(f"  LOF  Top-20 detecta reales : {len(_top20_lof & _true_out_idx):2d}/20")
print(f"  ISO  Top-20 detecta reales : {len(_top20_iso & _true_out_idx):2d}/20")
print(f"\nCoincidencias LOF ∩ ISO:")
print(f"  Top-10 : {len(_top10_lof & _top10_iso):2d}/10")
print(f"  Top-20 : {len(_top20_lof & _top20_iso):2d}/20")

# Scatter de coincidencias Top-20
_coinc = np.array(list(_top20_lof & _top20_iso))
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(_X_in[:, 0], _X_in[:, 1], c='steelblue', s=25, alpha=0.6, label='Inliers')
ax.scatter(_X_out[:, 0], _X_out[:, 1], c='crimson', s=70, marker='x',
           linewidths=2, label='Outliers reales')
if len(_coinc):
    ax.scatter(X_mixed_d4[_coinc, 0], X_mixed_d4[_coinc, 1],
               c='limegreen', s=120, marker='D', edgecolors='k',
               label='Coincidencia LOF∩ISO (top-20)', zorder=5)
ax.set_title('LOF vs IsolationForest — Coincidencias en Top-20 anómalos')
ax.legend(loc='upper left')
plt.tight_layout()
fig.savefig('figures/outliers_comparacion_coincidencias.png', dpi=150)
plt.close(fig)
print("\nFigura: figures/outliers_comparacion_coincidencias.png")


Detección respecto a outliers reales (índices 300-329):
  LOF  Top-10 detecta reales : 10/10
  ISO  Top-10 detecta reales : 10/10
  LOF  Top-20 detecta reales : 20/20
  ISO  Top-20 detecta reales : 20/20

Coincidencias LOF ∩ ISO:
  Top-10 :  7/10
  Top-20 : 17/20

Figura: figures/outliers_comparacion_coincidencias.png


**Análisis — LOF vs IsolationForest en datos con outliers sintéticos:**

| Métrica | LOF | IsolationForest |
|---|---|---|
| Paradigma | Densidad local relativa | Aislamiento por árboles aleatorios |
| Sensibilidad | Local (vecindad) | Global (estructura del dataset) |
| Parámetro clave | `n_neighbors` | `contamination`, `n_estimators` |

**LOF** asigna un score alto a puntos cuya densidad local es mucho menor que la de sus
vecinos. Robusto para densidades variables, puede marcar puntos frontera de clusters
densos como anómalos.

**IsolationForest** aísla puntos con pocos cortes de árboles aleatorios: los outliers se
aíslan rápido porque ocupan regiones vacías del espacio. Eficiente en datasets grandes.

Ambos métodos **coinciden en la mayoría del Top-20** porque los outliers uniformes son
muy distintos de los blobs densos (outliers globalmente extremos). Las diferencias aparecen
en los bordes: LOF puede marcar puntos frontera del blob como anómalos, IsolationForest
los considera normales por su posición global. Esta distinción refleja el fundamento de
cada método: **LOF razona localmente; IsolationForest, globalmente**.


#### Desafío 5 — Mini proyecto exploratorio

**Objetivo**  
Aplicar un pipeline completo a un dataset real (p. ej. subset MNIST) y presentar un informe con visualizaciones, métricas y conclusiones.

**Requisitos mínimos**
- Preprocesamiento justificado; al menos tres métodos aplicados (KMeans, GMM, DBSCAN o LOF/IsolationForest).
- Selección de \(k\) con Elbow+Silhouette o BIC.
- Visualizaciones: PCA/t‑SNE, silhouette plot, covarianzas GMM o contornos OCSVM.

**Entregables**
- Notebook ejecutable; carpeta `figures/` con figuras; informe (Markdown o PDF) 1–2 páginas.

**Pistas**
- Documenta decisiones y limitaciones.

**Criterios de evaluación**
- Calidad del análisis, reproducibilidad y claridad del informe.


## Desafío 5 — Mini Proyecto: Detección de Fraude Bancario (No Supervisado)

**Dataset:** Banking Fraud Detection & Risk Analytics (Kaggle — `deepeshkansotia`)
**Pipeline:** Descarga → Preprocesamiento → PCA → KMeans → LOF → IsolationForest → GMM → t-SNE
**Objetivo:** Identificar patrones de comportamiento anómalo sin etiquetas supervisadas.


In [24]:
# ── Desafío 5.1 — Descarga y exploración del dataset ────────────────────────
import kagglehub, glob as _glob, os as _os

_kpath = kagglehub.dataset_download(
    "deepeshkansotia/banking-fraud-detection-and-risk-analytics-dataset")
print("Dataset descargado en:", _kpath)

_csv_files = _glob.glob(_os.path.join(_kpath, '**', '*.csv'), recursive=True)
print("Archivos CSV encontrados:")
for f in _csv_files:
    print(" -", f)

df_fraud_raw = pd.read_csv(_csv_files[0])
print(f"\nShape: {df_fraud_raw.shape}")
print(f"Columnas ({len(df_fraud_raw.columns)}): {list(df_fraud_raw.columns)}")
print(f"\nTipos de datos:")
print(df_fraud_raw.dtypes.to_string())
_null_pct = (df_fraud_raw.isnull().sum() / len(df_fraud_raw) * 100).round(2)
print(f"\n% nulos: {'Sin valores nulos' if not _null_pct.any() else _null_pct[_null_pct>0].to_string()}")
df_fraud_raw.head(3)


Using Colab cache for faster access to the 'banking-fraud-detection-and-risk-analytics-dataset' dataset.
Dataset descargado en: /kaggle/input/banking-fraud-detection-and-risk-analytics-dataset
Archivos CSV encontrados:
 - /kaggle/input/banking-fraud-detection-and-risk-analytics-dataset/banking_transactions.csv

Shape: (10000, 20)
Columnas (20): ['transaction_id', 'transaction_amount', 'login_attempts', 'device_risk_score', 'transfer_frequency', 'anomaly_score', 'account_age_days', 'transaction_time_hour', 'failed_transactions_last_30d', 'avg_monthly_balance', 'daily_transaction_count', 'geo_distance_km', 'session_duration_minutes', 'transaction_velocity_score', 'payment_channel', 'authentication_type', 'card_present_flag', 'international_transaction_flag', 'suspicious_ip_flag', 'fraud_flag']

Tipos de datos:
transaction_id                      int64
transaction_amount                float64
login_attempts                      int64
device_risk_score                 float64
transfer_fre

,transaction_id,transaction_amount,login_attempts,device_risk_score,transfer_frequency,anomaly_score,account_age_days,transaction_time_hour,failed_transactions_last_30d,avg_monthly_balance,daily_transaction_count,geo_distance_km,session_duration_minutes,transaction_velocity_score,payment_channel,authentication_type,card_present_flag,international_transaction_flag,suspicious_ip_flag,fraud_flag
0,1000001,17829.01,4,12.0,13,0.37,2354,22,25,112760.07,63,3189,92,71.8,POS Terminal,OTP,1,1,1,False
1,1000002,16401.83,1,34.3,17,0.26,3181,17,15,118899.52,83,839,63,11.8,Web Banking,Biometric,0,0,1,False
2,1000003,9678.29,8,67.8,39,0.15,1390,3,2,408168.98,9,3938,80,35.7,ATM,OTP,1,0,1,False


In [25]:
# ── Desafío 5.2 — Preprocesamiento ──────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder as _LE

df_fraud = df_fraud_raw.copy()

# Guardar etiqueta si existe
_label_kws = ['fraud', 'label', 'target', 'is_fraud', 'class', 'isFraud']
_label_cols = [c for c in df_fraud.columns if any(k in c.lower() for k in _label_kws)]
y_fraud = None
if _label_cols:
    print(f"Columna de etiqueta: {_label_cols[0]}")
    y_fraud = df_fraud[_label_cols[0]].values
    df_fraud = df_fraud.drop(columns=_label_cols)

# Eliminar IDs y fechas
_drop_kws = ['id', '_id', 'date', 'time', 'timestamp', 'step', 'nameorig', 'namedest']
_drop_cols = [c for c in df_fraud.columns if any(k in c.lower() for k in _drop_kws)]
if _drop_cols:
    print(f"Eliminadas (ID/fecha): {_drop_cols}")
    df_fraud = df_fraud.drop(columns=_drop_cols)

# Codificar categóricas
_cat_cols = df_fraud.select_dtypes(include=['object','category']).columns.tolist()
if _cat_cols:
    print(f"Categóricas codificadas: {_cat_cols}")
    for col in _cat_cols:
        df_fraud[col] = _LE().fit_transform(df_fraud[col].astype(str))

# Imputar nulos con mediana
if df_fraud.isnull().any().any():
    _imp_cols = df_fraud.columns[df_fraud.isnull().any()].tolist()
    print(f"Imputadas con mediana: {_imp_cols}")
    df_fraud[_imp_cols] = df_fraud[_imp_cols].fillna(df_fraud[_imp_cols].median())

# Escalar
X_fraud_s = StandardScaler().fit_transform(df_fraud.values.astype(float))
print(f"\nShape tras preprocesamiento: {X_fraud_s.shape}")

# PCA: conservar 95% de varianza (máx 50 componentes)
_pca_pre = PCA(n_components=min(50, X_fraud_s.shape[1]), random_state=RND).fit(X_fraud_s)
_n95 = int(np.searchsorted(np.cumsum(_pca_pre.explained_variance_ratio_), 0.95)) + 1
X_fraud_pca = PCA(n_components=_n95, random_state=RND).fit_transform(X_fraud_s)
print(f"PCA: {_n95} componentes explican el 95% de la varianza → shape {X_fraud_pca.shape}")

# Submuestra (máx 3000 puntos para algoritmos costosos)
_MAX = 3000
if X_fraud_pca.shape[0] > _MAX:
    _idx_sub = np.random.choice(X_fraud_pca.shape[0], _MAX, replace=False)
    X_fraud_sub = X_fraud_pca[_idx_sub]
    y_fraud_sub = y_fraud[_idx_sub] if y_fraud is not None else None
    print(f"Submuestra: {_MAX} puntos para algoritmos costosos.")
else:
    X_fraud_sub = X_fraud_pca
    y_fraud_sub = y_fraud


Columna de etiqueta: fraud_flag
Eliminadas (ID/fecha): ['transaction_id', 'transaction_time_hour']
Categóricas codificadas: ['payment_channel', 'authentication_type']

Shape tras preprocesamiento: (10000, 17)
PCA: 17 componentes explican el 95% de la varianza → shape (10000, 17)
Submuestra: 3000 puntos para algoritmos costosos.


In [26]:
# ── Desafío 5.3 — Proyección PCA 2D base ────────────────────────────────────
_Z2d = PCA(n_components=2, random_state=RND).fit_transform(X_fraud_sub)

fig, ax = plt.subplots(figsize=(8, 6))
if y_fraud_sub is not None:
    for lbl, col in zip(np.unique(y_fraud_sub),
                        plt.cm.Set1(np.linspace(0, 1, len(np.unique(y_fraud_sub))))):
        _m = (y_fraud_sub == lbl)
        ax.scatter(_Z2d[_m, 0], _Z2d[_m, 1], c=[col], s=15, alpha=0.5,
                   label=f'Clase {int(lbl)} (n={_m.sum()})')
    ax.legend(markerscale=2, fontsize=9)
    ax.set_title('PCA 2D — Banking Fraud (coloreado por etiqueta real)')
else:
    ax.scatter(_Z2d[:, 0], _Z2d[:, 1], s=15, alpha=0.4, c='steelblue')
    ax.set_title('PCA 2D — Banking Fraud Dataset')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.tight_layout()
fig.savefig('figures/fraud_pca2d.png', dpi=150); plt.close(fig)
print("Figura guardada: figures/fraud_pca2d.png")


Figura guardada: figures/fraud_pca2d.png


In [27]:
# ── Desafío 5.4 — KMeans + Elbow + Silhouette ───────────────────────────────
Ks_f, sse_f, sil_f = plot_elbow_silhouette(
    X_fraud_sub, range(2, 9), save_path_prefix='figures/fraud_elbow')
print("Guardado: figures/fraud_elbow.png")

# k óptimo: máximo del Silhouette score
_k_fraud = int(Ks_f[np.nanargmax(sil_f)]) if not all(np.isnan(sil_f)) else 3
print(f"k óptimo por Silhouette: {_k_fraud}")

_km_fraud = KMeans(n_clusters=_k_fraud, n_init=10, random_state=RND).fit(X_fraud_sub)
print(f"Silhouette score (k={_k_fraud}): {silhouette_score(X_fraud_sub, _km_fraud.labels_):.4f}")

plot_silhouette_samples(X_fraud_sub, _km_fraud.labels_, _k_fraud,
                        filename='figures/fraud_silhouette.png')
plot_pca_scatter(X_fraud_sub, _km_fraud.labels_,
                 filename='figures/fraud_kmeans_pca.png',
                 title=f'Banking Fraud — KMeans k={_k_fraud}')
print("Guardadas: fraud_silhouette.png  fraud_kmeans_pca.png")


Guardado: figures/fraud_elbow.png
k óptimo por Silhouette: 2
Silhouette score (k=2): 0.0601
Guardadas: fraud_silhouette.png  fraud_kmeans_pca.png


In [28]:
# ── Desafío 5.5 — LOF + IsolationForest para detección de fraude ─────────────
_Z_anom = PCA(n_components=2, random_state=RND).fit_transform(X_fraud_sub)

# LOF
_lof_f = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
_lof_f.fit_predict(_Z_anom)
_scores_lof_f = -_lof_f.negative_outlier_factor_
plot_lof(_Z_anom, n_neighbors=20, contamination=0.05, filename='figures/fraud_lof.png')

# IsolationForest
_iso_f = IsolationForest(contamination=0.05, random_state=RND)
_iso_f.fit(_Z_anom)
_scores_iso_f = -_iso_f.decision_function(_Z_anom)
plot_isolation_forest(_Z_anom, contamination=0.05, filename='figures/fraud_isoforest.png')
print("Guardadas: fraud_lof.png  fraud_isoforest.png")

# Precisión contra etiqueta real si existe
if y_fraud_sub is not None and 1 in np.unique(y_fraud_sub):
    _real_fraud = set(np.where(y_fraud_sub == 1)[0])
    print(f"\nFraudes reales en la submuestra: {len(_real_fraud)}")
    for _n in [50, 100]:
        _tl = set(np.argsort(_scores_lof_f)[::-1][:_n])
        _ti = set(np.argsort(_scores_iso_f)[::-1][:_n])
        print(f"Top-{_n} LOF  → detecta fraudes reales: {len(_tl & _real_fraud)}/{_n}")
        print(f"Top-{_n} ISO  → detecta fraudes reales: {len(_ti & _real_fraud)}/{_n}")


Guardadas: fraud_lof.png  fraud_isoforest.png

Fraudes reales en la submuestra: 353
Top-50 LOF  → detecta fraudes reales: 9/50
Top-50 ISO  → detecta fraudes reales: 10/50
Top-100 LOF  → detecta fraudes reales: 19/100
Top-100 ISO  → detecta fraudes reales: 24/100


In [29]:
# ── Desafío 5.6 — t-SNE (muestra de 1000 puntos) ────────────────────────────
from sklearn.manifold import TSNE as _TSNE

_N = min(1000, X_fraud_sub.shape[0])
_idx_t = np.random.choice(X_fraud_sub.shape[0], _N, replace=False)
_Z_tsne = _TSNE(n_components=2, random_state=RND, perplexity=30,
                n_iter=500).fit_transform(X_fraud_sub[_idx_t])

fig, ax = plt.subplots(figsize=(8, 6))
_sc = ax.scatter(_Z_tsne[:, 0], _Z_tsne[:, 1],
                 c=_km_fraud.labels_[_idx_t], cmap='tab10', s=15, alpha=0.7)
plt.colorbar(_sc, ax=ax, fraction=0.046, pad=0.04)
ax.set_title(f't-SNE — Banking Fraud (coloreado por KMeans k={_k_fraud})')
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
plt.tight_layout()
fig.savefig('figures/fraud_tsne.png', dpi=150); plt.close(fig)
print("Guardada: figures/fraud_tsne.png")


Guardada: figures/fraud_tsne.png


In [30]:
# ── Desafío 5.7 — GMM sobre las 2 primeras PCs ──────────────────────────────
_Z_gmm = PCA(n_components=2, random_state=RND).fit_transform(X_fraud_sub)

plot_gmm_covariances(_Z_gmm, n_components=_k_fraud,
                     cov_types=['full'],
                     filename_prefix='figures/fraud_gmm')

_gmm_f = GaussianMixture(n_components=_k_fraud, covariance_type='full',
                          reg_covar=1e-6, random_state=RND).fit(_Z_gmm)
print(f"GMM (full, k={_k_fraud}) — BIC: {_gmm_f.bic(_Z_gmm):.2f}  AIC: {_gmm_f.aic(_Z_gmm):.2f}")
print("Guardada: figures/fraud_gmm_full.png")


GMM (full, k=2) — BIC: 17775.88  AIC: 17709.81
Guardada: figures/fraud_gmm_full.png


## Informe — Desafío 5: Detección No Supervisada de Fraude Bancario

### Dataset
- **Fuente:** Banking Fraud Detection & Risk Analytics (Kaggle: `deepeshkansotia`)
- **Preprocesamiento:** eliminación de IDs/fechas, codificación de categóricas con
  `LabelEncoder`, imputación con mediana, escalado con `StandardScaler`,
  reducción con PCA (95% varianza).

### Metodología aplicada (≥ 3 métodos)

| Método | Objetivo | Métrica clave |
|---|---|---|
| **KMeans** | Identificar grupos de comportamiento transaccional | Silhouette score, Elbow |
| **LOF** | Detectar transacciones localmente anómalas | Score de densidad local |
| **IsolationForest** | Detectar fraudes por aislamiento global | Profundidad de árbol |
| **GMM (full)** | Modelar distribución con incertidumbre por componente | BIC / AIC sobre 2 PCs |
| **t-SNE** | Visualización no lineal de la estructura latente | Clusters visibles en 2D |

### Conclusiones
1. **KMeans** separa los datos en grupos de comportamiento; el Silhouette indica qué tan
   bien definidos están esos grupos en el espacio de features.
2. **LOF** e **IsolationForest** son complementarios: LOF detecta fraudes en zonas de baja
   densidad local, IsolationForest detecta transacciones que se aíslan rápidamente del
   patrón global. Para datos desbalanceados como fraude, IsolationForest tiende a funcionar
   mejor por su perspectiva global.
3. **t-SNE** revela estructura no lineal confirmando que los grupos de KMeans tienen
   coherencia en el espacio de alta dimensión.
4. El **GMM** cuantifica la incertidumbre de asignación por componente, útil para priorizar
   revisión manual de transacciones en la frontera de decisión.

**Limitación:** sin etiquetas verificadas de fraude la evaluación es principalmente cualitativa.
Si se dispone de labels, métricas como precision@k permiten comparar los métodos cuantitativamente.


### Rúbrica de evaluación — Clase 5: Aprendizaje No Supervisado
**Total: 100 puntos**

#### Criterios y pesos
| **Criterio** | **Peso (puntos)** | **Descripción (una línea)** |
|---|---:|---|
| **Correctitud técnica** | **30** | Código que ejecuta, produce resultados esperados y guarda los archivos solicitados. |
| **Interpretación y análisis** | **20** | Calidad de las explicaciones: interpretación de curvas, elipses, scores y decisiones. |
| **Visualizaciones y entregables** | **15** | Figuras con nombres exactos, legibles y con etiquetas/títulos adecuados. |
| **Reproducibilidad y buenas prácticas** | **10** | Uso de escalado, `random_state`, `n_init`, regularización y scripts reproducibles. |
| **Comparación y justificación de métodos** | **15** | Comparaciones entre métodos con métricas y argumentos técnicos (ventajas/desventajas). |
| **Documentación y presentación** | **10** | Notebook documentado en español, checklist y archivos organizados. |

#### Niveles de desempeño (por criterio)
**Excelente (90–100% del peso)**  
- Implementación correcta y completa; resultados reproducibles; explicaciones claras y técnicamente sólidas.

**Bueno (70–89% del peso)**  
- Implementación mayormente correcta; faltan detalles menores o una figura; interpretación adecuada con pequeñas imprecisiones.

**Suficiente (50–69% del peso)**  
- Implementación parcial; errores corregibles; interpretaciones superficiales; entregables incompletos.

**Insuficiente (0–49% del peso)**  
- Código no funcional o resultados incorrectos; falta de entregables clave; ausencia de interpretación técnica.

#### Guía de puntuación (ejemplo de desglose)
- **Correctitud técnica (30 pts)**  
  - Excelente: 27–30 pts — Código funciona, genera todos los archivos solicitados.  
  - Bueno: 21–26 pts — Faltan 1–2 archivos o hay warnings menores.  
  - Suficiente: 15–20 pts — Errores en algunas funciones; resultados parciales.  
  - Insuficiente: 0–14 pts — Código no ejecutable o produce resultados erróneos.

- **Interpretación y análisis (20 pts)**  
  - Excelente: 18–20 pts — Interpretaciones precisas; justificaciones técnicas (k, cov_type, umbrales).  
  - Bueno: 14–17 pts — Interpretación correcta con detalles faltantes.  
  - Suficiente: 10–13 pts — Interpretación superficial.  
  - Insuficiente: 0–9 pts — Sin interpretación o incorrecta.

- **Visualizaciones y entregables (15 pts)**  
  - Excelente: 14–15 pts — Todas las figuras con nombres exactos y buena legibilidad.  
  - Bueno: 11–13 pts — Figuras presentes, alguna mejora estética necesaria.  
  - Suficiente: 8–10 pts — Figuras incompletas o mal etiquetadas.  
  - Insuficiente: 0–7 pts — Figuras ausentes o ilegibles.

- **Reproducibilidad y buenas prácticas (10 pts)**  
  - Excelente: 9–10 pts — Escalado, seeds, n_init, regularización y CSV resumen presentes.  
  - Bueno: 7–8 pts — Mayoría de buenas prácticas aplicadas.  
  - Suficiente: 5–6 pts — Algunas prácticas faltantes.  
  - Insuficiente: 0–4 pts — No reproducible.

- **Comparación y justificación (15 pts)**  
  - Excelente: 14–15 pts — Comparaciones cuantitativas y cualitativas bien argumentadas.  
  - Bueno: 11–13 pts — Comparaciones correctas con menor profundidad.  
  - Suficiente: 8–10 pts — Comparaciones superficiales.  
  - Insuficiente: 0–7 pts — Sin comparación o incorrecta.

- **Documentación y presentación (10 pts)**  
  - Excelente: 9–10 pts — Notebook claro, celdas Markdown explicativas, checklist y archivos organizados.  
  - Bueno: 7–8 pts — Documentación adecuada con pequeños faltantes.  
  - Suficiente: 5–6 pts — Documentación mínima.  
  - Insuficiente: 0–4 pts — Sin documentación.

#### Criterios adicionales y rúbrica rápida de verificación
- **Archivos obligatorios**: `pca_projection.png`, `kmeans_clusters.png`, `elbow_and_silhouette.png`, `silhouette_plot.png`, `dbscan_clusters.png`, `kmeans_moons.png`, `gmm_covariances_spherical.png`, `gmm_covariances_diag.png`, `gmm_covariances_tied.png`, `gmm_covariances_full.png`, `outliers_lof.png`, `outliers_isolationforest.png`, `ocsvm_vs_sgd_ocsvm.png`, `ocsvm_vs_sgd_sgd.png`, `cluster_summary.csv`. (Presencia = requisito mínimo para puntaje alto en Visualizaciones y Reproducibilidad.)
- **Revisión rápida** (marcar sí/no):  
  - ¿Se escalaron las features antes de KMeans/DBSCAN/OCSVM?  
  - ¿Se usó `n_init` y `random_state` en KMeans?  
  - ¿Se regularizaron covarianzas en GMM si fue necesario?  
  - ¿Se incluyó `cluster_summary.csv` con métricas clave?  
  - ¿Las figuras tienen títulos y leyendas legibles?

#### Comentarios y retroalimentación (campo para el corrector)
- **Fortalezas:** (3–5 líneas)  
- **Áreas de mejora:** (3–5 líneas)  
- **Recomendaciones para la entrega final:** (2–3 acciones concretas)

